In [ ]:
import os
import pickle
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

from wordcloud import WordCloud


In [ ]:
GLOBAL_CMAP = 'tab10'

In [ ]:
# Load word data
processed_dir = os.path.join('..', 'data', 'processed')

splits = {}
for split_name in ['train', 'val', 'test']:
    path = os.path.join(processed_dir, f'words_{split_name}.pkl')
    with open(path, 'rb') as f:
        splits[split_name] = pickle.load(f)

df_all   = pd.concat(splits.values(), ignore_index=True)
df_train = splits['train']
df_test  = splits['test']

print(f"Loaded {len(df_all)} total scripts across all splits")
df_all[['nominated', 'winner', 'token_count']].describe().style.format({
    'nominated':   '{:.3f}',
    'winner':      '{:.3f}',
    'token_count': '{:.2f}',
})

In [ ]:
# Global wordcloud across all splits


font_path = font_manager.findfont("DejaVu Serif")

all_tokens = ' '.join(token for tokens in df_all['tokens'] for token in tokens)

wc_global = WordCloud(
    width=1600,
    height=800,
    background_color='white',
    max_words=200,
    collocations=False,
    color_func=lambda *args, **kwargs: "black", font_path=font_path
).generate(all_tokens)

fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(wc_global, interpolation='bilinear')
ax.axis('off')
ax.set_title('Global Word Cloud — All Splits', fontsize=18, pad=16)
plt.tight_layout()
plt.show()

In [ ]:
TOP_N        = 100
DISPLAY_N    = 10
EPS          = 1e-9
MIN_DOC_FREQ = 0.30


In [ ]:
# Per-split wordclouds (train / val / test) side by side

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for ax, (split_name, df_split) in zip(axes, splits.items()):
    split_tokens = ' '.join(token for tokens in df_split['tokens'] for token in tokens)

    wc = WordCloud(
        width=800,
        height=600,
        background_color='white',
        max_words=150,
        collocations=False,
        color_func=lambda *args, **kwargs: "black", font_path=font_path
    ).generate(split_tokens)

    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'{split_name.upper()} ({len(df_split)} scripts)', fontsize=14, pad=12)

plt.suptitle('Word Clouds by Split', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Keyness analysis hyperparameters
TOP_N        = 100   # number of top keywords to keep per side
DISPLAY_N    = 10    # how many to show in tables / wordclouds

EPS          = 1e-9  # additive smoothing constant for the relative-risk ratio
                     # (handles zero-frequency words; cf. Laplace smoothing)
MIN_DOC_FREQ = 0.30  # minimum range/dispersion: a word must appear in >=X%
                     # of documents in its target sub-corpus to be eligible.
                     # Filters out rare terms (e.g. character names) that
                     # would otherwise dominate the keyword list.

WC_KWARGS = dict(width=900, height=600, background_color='white',
                 max_words=TOP_N, collocations=False,
                 color_func=lambda *args, **kwargs: "black", font_path=font_path)

## Keyness Analysis

The next sections perform what corpus linguists call **keyness analysis** —
identifying words that are characteristically over-represented in one
sub-corpus (the *target*) relative to another (the *reference*). Here the
two corpora are scripts of **Oscar-nominated** vs. **non-nominated** films.

**Method.** For each word $w$ we compute its relative frequency in each
corpus and take the ratio:

$$\text{ratio}(w) = \frac{\text{rf}_{\text{target}}(w)}{\text{rf}_{\text{reference}}(w)}$$

This is the **relative risk** measure (a.k.a. *ratio metric*, Kilgarriff 2009) that could help **distinguish** one class of scripts compared to a reference group.

**Smoothing.** `EPS = 1e-9` handles the zero-frequency problem (a word absent
from one sub-corpus would otherwise blow up the ratio). This is essentially
additive smoothing.

**Range filter.** `MIN_DOC_FREQ = 0.30` requires a word to appear in ≥30% of
documents in its corpus before it's eligible — this is the **range** /
**dispersion** filter, and it drops rare keywords that show up many times
in one or two scripts (e.g., a unique character name) but aren't characteristic
of the corpus as a whole.

In [ ]:
def keyness_tables(df, eps=EPS, min_doc_freq=MIN_DOC_FREQ, top_n=TOP_N):
    """Compute keyness tables comparing nominated (target) vs non-nominated
    (reference) sub-corpora.

    Returns (top_nom, top_non, df_words, nom, non) where the top tables are
    ranked by the *relative risk* effect-size measure (ratio of relative
    frequencies), filtered by document-frequency (range) >= min_doc_freq.
    """
    # Split into target (nominated) and reference (non-nominated) corpora.
    nom = df[df['nominated'] == 1]
    non = df[df['nominated'] == 0]

    def relative_freqs(subset):
        # Normalized term frequency: count / total tokens in the corpus.
        # In corpus-linguistics terms this is the per-corpus "rf" — sometimes
        # reported per million words (pmw) but here kept as a raw proportion.
        counts = Counter(t for tokens in subset['tokens'] for t in tokens)
        total  = sum(counts.values())
        return {w: c / total for w, c in counts.items()}

    rf_nom = relative_freqs(nom)
    rf_non = relative_freqs(non)

    # Document frequency (a.k.a. "range" or "dispersion"): the fraction of
    # documents in each sub-corpus that contain the word at least once. Used
    # below to filter out rare words — terms that get high ratios because
    # they appear many times in one or two scripts (character names, place
    # names, etc.) rather than being broadly characteristic of the corpus.
    n_nom_docs  = len(nom)
    n_non_docs  = len(non)
    df_freq_nom = defaultdict(int)
    df_freq_non = defaultdict(int)
    for tokens in nom['tokens']:
        for w in set(tokens):
            df_freq_nom[w] += 1
    for tokens in non['tokens']:
        for w in set(tokens):
            df_freq_non[w] += 1

    # Build the keyness table. Columns:
    #   rf_nom, rf_non    — relative frequency in target / reference
    #   ratio_nom         — relative risk with NOM as target (rf_nom / rf_non)
    #   ratio_non         — relative risk with NON as target (rf_non / rf_nom)
    #   df_nom, df_non    — document-frequency (range) in each sub-corpus
    #
    # `eps` provides additive smoothing so that words absent from one corpus
    # don't produce 0/0 or division-by-zero. It also caps the maximum ratio
    # for very rare words at ~rf / eps.
    all_words = set(rf_nom) | set(rf_non)
    rows = [
        {
            'word':      w,
            'rf_nom':    rf_nom.get(w, 0),
            'rf_non':    rf_non.get(w, 0),
            'ratio_nom': rf_nom.get(w, eps) / rf_non.get(w, eps),
            'ratio_non': rf_non.get(w, eps) / rf_nom.get(w, eps),
            'df_nom':    df_freq_nom[w] / n_nom_docs,
            'df_non':    df_freq_non[w] / n_non_docs,
        }
        for w in all_words
    ]
    df_words = pd.DataFrame(rows)

    # Apply the range filter on the *target* side: a word only counts as a
    # keyword for NOMINATED if it appears in at least `min_doc_freq` of the
    # nominated scripts. Then rank by relative risk and take top_n.
    top_nom = (
        df_words[df_words['df_nom'] >= min_doc_freq]
        .nlargest(top_n, 'ratio_nom')[['word', 'rf_nom', 'ratio_nom']]
        .rename(columns={'rf_nom': 'rel_freq_in_nominated', 'ratio_nom': 'ratio_vs_non'})
        .reset_index(drop=True)
    )
    top_non = (
        df_words[df_words['df_non'] >= min_doc_freq]
        .nlargest(top_n, 'ratio_non')[['word', 'rf_non', 'ratio_non']]
        .rename(columns={'rf_non': 'rel_freq_in_non_nominated', 'ratio_non': 'ratio_vs_nom'})
        .reset_index(drop=True)
    )
    return top_nom, top_non, df_words, nom, non

In [ ]:
top_nom_global, top_non_global, _, nom_global, non_global = keyness_tables(df_all)

print(f"GLOBAL  |  {len(nom_global)} nominated  |  {len(non_global)} non-nominated\n")

print(f"Top {TOP_N} keywords more frequent in NOMINATED (showing {DISPLAY_N}):")
display(top_nom_global.head(DISPLAY_N).style.format(
    {'rel_freq_in_nominated': '{:.6f}', 'ratio_vs_non': '{:.2f}'}
))

print(f"\nTop {TOP_N} keywords more frequent in NON-NOMINATED (showing {DISPLAY_N}):")
display(top_non_global.head(DISPLAY_N).style.format(
    {'rel_freq_in_non_nominated': '{:.6f}', 'ratio_vs_nom': '{:.2f}'}
))

In [ ]:
wc_nom_g = WordCloud(**WC_KWARGS).generate_from_frequencies(
    dict(zip(top_nom_global['word'], top_nom_global['rel_freq_in_nominated']))
)
wc_non_g = WordCloud(**WC_KWARGS).generate_from_frequencies(
    dict(zip(top_non_global['word'], top_non_global['rel_freq_in_non_nominated']))
)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
axes[0].imshow(wc_nom_g, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title('Distinguishing keywords for NOMINATED (global)', fontsize=13, pad=12)
axes[1].imshow(wc_non_g, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title('Distinguishing keywords for NON-NOMINATED (global)', fontsize=13, pad=12)

plt.suptitle(f'Top {TOP_N} Distinguishing Keywords — GLOBAL', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## Keyness Analysis — By Split

In [ ]:
# Compute tables for all three splits
split_results = {
    name: keyness_tables(splits[name])
    for name in ['train', 'val', 'test']
}

In [ ]:
from IPython.display import display, HTML

for split_name in ['train', 'val', 'test']:
    top_nom, top_non, _, nom, non = split_results[split_name]

    left  = top_nom.head(DISPLAY_N).reset_index(drop=True)
    right = top_non.head(DISPLAY_N).reset_index(drop=True)

    left_html  = left.style.format({'rel_freq_in_nominated':     '{:.6f}', 'ratio_vs_non': '{:.2f}'}).to_html()
    right_html = right.style.format({'rel_freq_in_non_nominated': '{:.6f}', 'ratio_vs_nom': '{:.2f}'}).to_html()

    combined = f"""
    <h4>{split_name.upper()} — {len(nom)} nominated | {len(non)} non-nominated</h4>
    <table style="width:100%"><tr>
        <td style="width:50%; vertical-align:top; padding-right:20px">
            <b>More frequent in NOMINATED</b><br>{left_html}
        </td>
        <td style="width:50%; vertical-align:top">
            <b>More frequent in NON-NOMINATED</b><br>{right_html}
        </td>
    </tr></table><hr>
    """
    display(HTML(combined))

In [ ]:
# 3x2 grid: one row per split, nominated | non-nominated
fig, axes = plt.subplots(3, 2, figsize=(22, 22))
plt.subplots_adjust(hspace=0.25, wspace=0.01)

for row, split_name in enumerate(['train', 'val', 'test']):
    top_nom, top_non, _, nom, non = split_results[split_name]

    wc_nom = WordCloud(**WC_KWARGS).generate_from_frequencies(
        dict(zip(top_nom['word'], top_nom['rel_freq_in_nominated']))
    )
    wc_non = WordCloud(**WC_KWARGS).generate_from_frequencies(
        dict(zip(top_non['word'], top_non['rel_freq_in_non_nominated']))
    )

    axes[row, 0].imshow(wc_nom, interpolation='bilinear')
    axes[row, 0].axis('off')
    axes[row, 0].set_title(f'Distinguishing keywords for NOMINATED — {split_name.upper()} ({len(nom)} scripts)', fontsize=12, pad=10)

    axes[row, 1].imshow(wc_non, interpolation='bilinear')
    axes[row, 1].axis('off')
    axes[row, 1].set_title(f'Distinguishing keywords for NON-NOMINATED — {split_name.upper()} ({len(non)} scripts)', fontsize=12, pad=10)

plt.suptitle(f'Top {TOP_N} Distinguishing Keywords by Split', fontsize=16, y=0.92)
plt.show()